# ESCUELA POLITECNICA NACIONAL

- Nombre: Freddy Jiménez
- Fecha: 29/06/2026

## EJERCICIO USO DE LA API DE GEMINI

In [ ]:
apy_key = "My_Apy_Key"  

In [4]:
import os
from google import genai

# Inicializa el cliente. 
API_KEY_GEMINI_FJ = apy_key

client = genai.Client(api_key=API_KEY_GEMINI_FJ)

# Probamos una respuesta básica
response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents='Hola Gemini, ¿cómo estás hoy?',
)

print(response.text)

¡Hola! Estoy muy bien, gracias por preguntar. Como modelo de lenguaje, no tengo sentimientos ni estados de ánimo en el sentido humano, pero estoy funcionando perfectamente y listo para ayudarte en lo que necesites.

¿En qué puedo asistirte hoy?


### RETRIEVAL. -

In [5]:
# Se carga el conjunto de datos 20 Newsgroups, que es un conjunto de datos de texto comúnmente
# utilizado para tareas de clasificación de texto. Este conjunto contiene aproximadamente 20,000
# documentos de noticias distribuidos en 20 categorías diferentes.

import pandas as pd
from sklearn.datasets import fetch_20newsgroups

# Se carga el conjunto de datos 20 Newsgroups, eliminando encabezados, 
# pies de página y citas para centrarse en el contenido principal de los documentos.
newsgroups_data = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))

# Se crea el dataframe de pandas con los documentos y sus categorías correspondientes.
df = pd.DataFrame({
    'documento': newsgroups_data.data,
    'categoria': [newsgroups_data.target_names[t] for t in newsgroups_data.target]
})

# Limpiamos un poco eliminando filas vacías o con solo espacios en blanco
df = df[df['documento'].str.strip() != '']
df = df.reset_index(drop=True)

# Para no agotar la cuota de la API , limitaremos el corpus a los primeros 100 documentos
df = df.head(100)

print(f"Corpus cargado con éxito. Total de documentos seleccionados: {len(df)}")
df.head()

Corpus cargado con éxito. Total de documentos seleccionados: 100


,documento,categoria
0,I was wondering if anyone out there could enli...,rec.autos
1,A fair number of brave souls who upgraded thei...,comp.sys.mac.hardware
2,"well folks, my mac plus finally gave up the gh...",comp.sys.mac.hardware
3,\nDo you have Weitek's address/phone number? ...,comp.graphics
4,"From article <C5owCB.n3p@world.std.com>, by to...",sci.space


### TRANSFORMACION A EMBEDDINGS. -

In [6]:
# En este bloque, generaremos embeddings para cada documento en el DataFrame utilizando el modelo "models/gemini-embedding-2".
# Lo cual nos permitirá representar cada documento como un vector numérico en un espacio de alta dimensión, 
# facilitando tareas como la búsqueda semántica y la clasificación de texto.

import time
import numpy as np

embeddings = []

print("Generando embeddings para los documentos...")
for i, texto in enumerate(df['documento']):
    try:
        # Usamos el modelo "models/gemini-embedding-2" para generar embeddings
        response = client.models.embed_content(
            model="models/gemini-embedding-2", 
            contents=texto
        )
        
        embedding_vector = response.embeddings[0].values
        embeddings.append(embedding_vector)
        
        # Se da un delay de 0.5 segundos entre cada solicitud para evitar sobrecargar la API
        time.sleep(0.5) 
        
    except Exception as e:
        print(f"Error en el documento índice {i}: {e}")
        break

# Guardamos en el dataframe los embeddings generados, 
# asegurándonos de que la cantidad de embeddings coincida con la cantidad de documentos.
if len(embeddings) == len(df):
    df['embedding'] = embeddings
    print("¡Embeddings generados y guardados con éxito en el DataFrame usando models/gemini-embedding-2!")
else:
    print(f"Ocurrió un problema. Se generaron {len(embeddings)} embeddings de {len(df)} documentos.")

Generando embeddings para los documentos...
¡Embeddings generados y guardados con éxito en el DataFrame usando models/gemini-embedding-2!


### CREACION DE UNA QUERY Y LA BUSQUEDA. -

In [9]:
# Ahora, vamos a definir una consulta de búsqueda y generar su embedding correspondiente 
# utilizando el mismo modelo "models/gemini-embedding-2". Esto nos permitirá comparar la 
# consulta con los embeddings de los documentos para encontrar los más relevantes.

import numpy as np

# Se define la consulta de búsqueda que queremos realizar.
query = "Cybersecurity threats and their impact on businesses"

# Usamos el modelo "models/gemini-embedding-2" para generar el embedding de la consulta.
response_query = client.models.embed_content(
    model="models/gemini-embedding-2",
    contents=query
)
query_embedding = np.array(response_query.embeddings[0].values)

print(f"Query definida: '{query}' y transformada a embedding con éxito usando models/gemini-embedding-2.")

Query definida: 'Cybersecurity threats and their impact on businesses' y transformada a embedding con éxito usando models/gemini-embedding-2.


In [10]:
# Para esta parte del código, utilizaremos la similitud de coseno para medir qué tan 
# similares son los embeddings de los documentos con respecto al embedding de la consulta.

from sklearn.metrics.pairwise import cosine_similarity

# Calculamos la similitud de coseno entre el embedding de la query y todos los del DataFrame
similitudes = cosine_similarity([query_embedding], np.vstack(df['embedding'].values))[0]

# Agregamos los scores de similitud al DataFrame
df['similitud'] = similitudes

# Ordenamos de mayor a menor similitud y tomamos los 5 mejores
top_5_resultados = df.sort_values(by='similitud', ascending=False).head(5)

In [ ]:
# Para esta parte del codigo, vamos a mostrar los resultados de la búsqueda, incluyendo la categoría original del documento,
# el score de similitud y un extracto del contenido del documento para que el usuario pueda evaluar rápidamente la relevancia 
# de cada resultado.

print(f"--- Top 5 documentos más similares para la búsqueda: '{query}' ---\n")

for idx, fila in top_5_resultados.iterrows():
    print(f"Ranking {top_5_resultados.index.get_loc(idx) + 1}")
    print(f"Categoría original: {fila['categoria']}")
    print(f"Score de Similitud: {fila['similitud']:.4f}")
    # Mostramos los primeros 250 caracteres del documento para no inundar la pantalla
    print(f"Contenido: {fila['documento'][:250].strip()}...")
    print("-" * 60)

--- Top 5 documentos más similares para la búsqueda: 'Cybersecurity threats and their impact on businesses' ---

Ranking 1
Categoría original: comp.sys.mac.hardware
Score de Similitud: 0.6128
Contenido: --...
------------------------------------------------------------
Ranking 2
Categoría original: sci.crypt
Score de Similitud: 0.6124
Contenido: Okay, let's suppose that the NSA/NIST/Mykotronix Registered
Key system becomes standard and I'm able to buy such a system
from my local radio shack. Every phone comes with a built in
chip and the government has the key to every phone call. 
I go and...
------------------------------------------------------------
Ranking 3
Categoría original: sci.crypt
Score de Similitud: 0.5682
Contenido: I sent a response to the White House at

	0005895485@MCIMAIL.COM (White House)

and received a nice, automatic reply from MICMAIL noting, in passing, that
if I had included a SNail address, I would get a reply in due course.

For those who care, my r...
------